# 03-01 — Cross-Site MLP Loss Curves (best-config reruns)

Standalone diagnostic next to ``03-CrossSite-Classifier.ipynb``. The main
notebook grid-searches lr x dropout per drug on A and reports BalAcc/AUC, but
does not persist the per-epoch loss history.

This notebook trains **one MLP per drug at the best hyperparameters** from
``07-.../best_params.csv`` on DRIAMS-A (species-stratified 80/20, same as 03)
and logs per-epoch metrics.

- Outputs: ``results_loss_curves_03/metrics_<drug>.csv`` and ``03_loss_curves.pdf``

Run on Colab (GPU recommended; 10 fits). Set env ``SMOKE=1`` for a dry run.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

if IN_COLAB:
    !pip install maldideepkit maldiamrkit --quiet

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from tqdm.auto import tqdm

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldideepkit.attention.mlp import MaldiMLPClassifier
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
np.random.seed(SEED)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
    REPO_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed/Processing/Analysis")
else:
    DATA_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed")
    REPO_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed/Processing/Analysis")

BEST_PARAMS_CSV = REPO_ROOT / "07-Dedicated-MLP-Aggregated/07-01-results_dedicated_lr_mlp/best_params.csv"

# Embedded copy of 07's best_params.csv (CSV is not tracked) — used if the file is absent.
DEFAULT_BEST = {
    "Ciprofloxacin": {"lr": 7.8e-05, "drop": 0.8},
    "Gentamicin": {"lr": 7.5e-05, "drop": 0.8},
    "Amoxicillin-Clavulan": {"lr": 7.5e-05, "drop": 0.7},
    "Piperacillin-Tazobac": {"lr": 9.2e-05, "drop": 0.8},
    "Cefepime": {"lr": 8.6e-05, "drop": 0.7},
    "Ceftriaxone": {"lr": 1.8e-04, "drop": 0.52},
    "Imipenem": {"lr": 7.5e-05, "drop": 0.8},
    "Ceftazidime": {"lr": 1.8e-04, "drop": 0.36},
    "Vancomycin": {"lr": 9.2e-05, "drop": 0.5},
    "Amikacin": {"lr": 8.4e-05, "drop": 0.5},
}
OUT_DIR = Path("./results_loss_curves_03")
OUT_DIR.mkdir(exist_ok=True)

TRAIN_SITE = "Proc_DRIAMS-A"
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS = [
    "Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic_acid",
    "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
    "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin",
]
if SMOKE:
    DRUGS = DRUGS[:2]

print(f"SMOKE={SMOKE} | drugs={len(DRUGS)} | data: {DATA_ROOT}")

In [ ]:
# ── Best hyperparameters from 07 ──
def load_best_mlp_params():
    if not BEST_PARAMS_CSV.exists():
        print(f"best_params.csv not found ({BEST_PARAMS_CSV}) — using embedded defaults from 07.")
        return dict(DEFAULT_BEST)
    df = pd.read_csv(BEST_PARAMS_CSV)
    out = {}
    for _, r in df.iterrows():
        mlp = str(r["MLP"])
        out[str(r["Drug"])] = {
            "lr": float(mlp.split("lr=")[1].split(" ")[0]),
            "drop": float(mlp.split("drop=")[1]),
        }
    return out

BEST = load_best_mlp_params()

def get_params(drug):
    for k, v in BEST.items():
        if drug.startswith(k) or k.startswith(drug):
            return v
    return {"lr": 7.8e-05, "drop": 0.8}

for d in DRUGS:
    print(f"{d:35s} -> {get_params(d)}")

In [ ]:
# ── Train one best-config MLP per drug on A ──
summary = []

for drug in tqdm(DRUGS, desc="Loss curves"):
    p = get_params(drug)
    fpath = DATA_ROOT / TRAIN_SITE / drug / "data.csv"
    if not fpath.exists():
        print(f"  {drug}: no A data, skipping")
        continue
    df = pd.read_csv(fpath)
    X = df[BIN_COLS].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype=int)
    species = df["species"].values
    if SMOKE:
        rs = np.random.RandomState(SEED)
        idx = rs.choice(len(y), size=min(1500, len(y)), replace=False)
        X, y, species = X[idx], y[idx], species[idx]

    idx = np.arange(len(y)).reshape(-1, 1)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx, y, species=species, test_size=0.20, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)

    state = fit_input_transform(X[idx_train], "log1p+standardize")
    X_train = apply_input_transform(X[idx_train], state)
    X_val = apply_input_transform(X[idx_val], state)
    y_train, y_val = y[idx_train], y[idx_val]

    tag = drug.replace("/", "_").replace(" ", "_").lower()
    log_path = OUT_DIR / f"metrics_{tag}.csv"
    if log_path.exists():
        log_path.unlink()

    mlp = MaldiMLPClassifier(
        hidden_dim=512, head_dims=(256, 128), use_attention=False,
        dropout_high=p["drop"], dropout_low=p["drop"] / 2.0, weight_decay=1e-4,
        learning_rate=p["lr"], batch_size=64,
        epochs=3 if SMOKE else 50,
        early_stopping_patience=2 if SMOKE else 10,
        warmup_epochs=0 if SMOKE else 10,
        val_fraction=0.1, use_sam=False,
        input_transform="none", tune_threshold=False,
        track_train_metrics=True, metrics_log_path=log_path,
        random_state=SEED, verbose=False)
    mlp.fit(X_train, y_train)

    proba = mlp.predict_proba(X_val)[:, 1]
    thr_grid = np.linspace(0.05, 0.95, 91)
    best_t = thr_grid[np.argmax([balanced_accuracy_score(y_val, proba >= t) for t in thr_grid])]
    ba = balanced_accuracy_score(y_val, proba >= best_t)
    auc = roc_auc_score(y_val, proba)
    n_epochs = len(pd.read_csv(log_path))
    summary.append({"drug": drug, "n_train": len(y_train), "n_val": len(y_val),
                    "lr": p["lr"], "dropout": p["drop"], "epochs_logged": n_epochs,
                    "val_balacc": ba, "val_auc": auc})
    print(f"  {drug:35s} epochs={n_epochs:3d}  val BalAcc={ba:.4f}  AUC={auc:.4f}")

df_summary = pd.DataFrame(summary)
df_summary.to_csv(OUT_DIR / "summary_loss_curves_03.csv", index=False)
df_summary

In [ ]:
# ── Plot per-drug train/val loss ──
log_paths = sorted(glob.glob(str(OUT_DIR / "metrics_*.csv")))
assert log_paths, "No metrics_*.csv found — run the training cell above first."

n = len(log_paths)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.6 * nrows), squeeze=False)
for ax, path in zip(axes.ravel(), log_paths):
    df = pd.read_csv(path)
    ep = df["epoch"] + 1
    ax.plot(ep, df["train_loss"], label="train", lw=1.6)
    ax.plot(ep, df["val_loss"], label="val", lw=1.6)
    best_ep = int(df.loc[df["val_loss"].idxmin(), "epoch"]) + 1
    ax.axvline(best_ep, color="gray", ls=":", lw=1)
    ax.set_title(Path(path).stem.replace("metrics_", ""), fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("CE loss")
    ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.4)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle("03 — cross-site MLP (trained on A) per-drug training curves",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "03_loss_curves.pdf", bbox_inches="tight")
plt.show()

## What to look for

- Per-drug early-stop epochs vs. the drug's resistance rate and sample size.
- Whether the A-only models overfit faster than the pooled models in 02
  (they see ~1/4 of the data).